# Main for Energy Simulation 

Simulates the energy consumption for each track, calculates the SoC each truck would have and calculates the resulting load profiles for the freight forwarding locations 

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and main_data_analysis.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns
import sys

sys.path.append('energy_sim')
import data.data_handling.data_visualization as dv
import data.data_handling.data_processing as dp
from data.data_handling.data_processing import colors_fleets, alpha_major, alpha_minor
from energy_sim.energy_sim_functions import faulty_tracks, run_energy_sim, clean_energy_sim_data
import data.data_handling.sequential_analysis as sq

## Perform Energy Simulation based on BET.OS script 

The energy simulation takes very long to run (in total approx. 24 h) and should just only be executed once 

First run the faulty_tracks() function to identify tracks with missing data (results will be saved in mismatched_tracks.csv)
Then run run_energy_sim() to calculate energy consumption for each track (results will be saved in tracks_filtered_with_energy.csv)
Run run_energy_sim() will additionally create zero_speed_tracks.csv which lists tracks that have no speed data

The outputs are saved as csvs

### Run once:

In [ ]:
#faulty_tracks()

# trips_energy = run_energy_sim(override_mismatched_tracks=False)

# "Can either pass on trips_energy directly from the simulation or let the function read in tracks_with_energy_raw.csv"
# df_trip_energy = clean_energy_sim_data(trips=None)

### After running the energy simulation once, read the csv instead of excecuting the simulation

In [ ]:
df_trip_energy= pd.read_csv('data/output/csvs/tracks_with_energy.csv', index_col='track_id')

In [ ]:
df_trips = pd.read_csv('input/stations/tracks_filtered.csv', index_col='track_id')
df_trips['stop_time'] = pd.to_datetime(df_trips['stop_time'], format='ISO8601')
df_trips['start_time'] = pd.to_datetime(df_trips['start_time'], format='ISO8601')

df_stops = dp.process_stops_data(df_trips)
df_occ_energy = sq.combine_tracks_and_stops(df_stops = df_stops, df_tracks_with_energy = df_trip_energy)

### Calculate Battery charges and SoCs at the end of each activity *

In [ ]:
# Constant charging powers 
charging_powers = {'home base': 50, 'industrial area': 350}
df_soc, public_charging = sq.truck_soc(df_activities = df_occ_energy, charging_powers = charging_powers, soc_min = 0.15)

### Tour statistics *

In [ ]:
"""
In contrast to df_tours, df_tours_energy also includes non-driving activities between tracks. 
Thus, some additional parameters are included

    Stop time is the stop time of the last track equivalently to df_tours, 
    End time additionally includes the time spent at the home base after the tour's last track (i.e. = start time of the next tour)
   
    driving_duration is the sum of the duration of all driving activities in the tour, equivalently to duration in df_tours
    duration is the time between the start of the tour's first track and the end of its last activity 
     (i.e the duration between the start of the tour and the start of the next tour)
"""

df_tours_energy = dp.aggregate_tours(df_soc, charging_powers, save=True, energy=True, min_soc=0.15)

In [ ]:
 # TODO adjust the plots into something meaninfull. Ideas:
# First See what the data looks like for different charging strategies. Then:
# 1) plot the amount (ratio of occurences to total tours) of SoCs below a certain threshold = 0.15
# 2) plot the distance travelled under the SoC threshold = 0.15
# 3) maybe try a cumulative density plot with SoC on x axis and the number of occurences on the y axis 
#    (SoC = 1 is the origen), up to threshold the plot is green, afterwards orange
# 4) try a histplot?? https://seaborn.pydata.org/examples/histogram_stacked.html 
#    (what could be important additional information? one plot with all ffs, or maybe length of tour?)

# TODO See alternative plot verion in notes for plot in the same format of weekly distance boxplot  
df_soc_plot = df_tours_energy.copy()
#result = dv.plot_soc_by_freight_forwarder_symlog(df_soc_plot)
result = dv.plot_soc_by_freight_forwarder(df_tours_energy = df_soc_plot, charging_powers=charging_powers)

## Freight Forwarding Location Analysis

### Energy recharged potential of each tour on its own
In contrast to the energy_recharged_kwh and energy_recharged_kwh_potential calculated by sq.truck_soc(), this function only considers the energy that can be recharged within a tour, i.e disregarding truck disposition. This is equivalent to the assumption that the SoC at the start of each tour = SoC max. 

In [ ]:
tracks_energy_no_disp = dp.tracks_energy_con_and_regen(df_activities = df_occ_energy, charging_powers = charging_powers, soc_min=0.15)

### Tour statistics *

In [ ]:
df_tours_energy_no_disp = dp.aggregate_tours(tracks_energy_no_disp, charging_powers, save=False, energy=True, min_soc=0.15)
df_tours_energy_no_disp.to_csv(f"data/output/track_energies/tours_constant_charging_{charging_powers['home base']}-{charging_powers['industrial area']}_no_disp.csv", index=False)

""" Print some statistics """
# Calculate the threshold
Battery_capacity = 572  # in kWh
max_dod = 0.75  # maximum depth of discharge
threshold = Battery_capacity * max_dod

# Tours with too high energy consumption
problematic_tours = df_tours_energy_no_disp[df_tours_energy_no_disp['energy_consumption_kwh_cleaned'] - df_tours_energy_no_disp['energy_recharged_kwh'] > threshold]
problematic_tours_wo_indu = df_tours_energy_no_disp[df_tours_energy_no_disp['energy_consumption_kwh_cleaned'] > threshold]


# Calculate absolute and percentage values
absolute_count = problematic_tours.shape[0]
percentage_count = (absolute_count / df_tours_energy_no_disp.shape[0]) * 100

# Print the results
print(f"Absolute number of tours with too high energy consumption: {absolute_count}")
print(f"Percentage of total tours: {percentage_count:.2f}%")

print(f"Absolute number of tours with too high energy consumption (without industrial area charging): {problematic_tours_wo_indu.shape[0]}")
print(f"Percentage of total tours (without industrial area charging): {(problematic_tours_wo_indu.shape[0] / df_tours_energy_no_disp.shape[0]) * 100:.2f}%")

print('\n')
energy_industrial = df_tours_energy_no_disp['energy_recharged_kwh'].sum()
required_energy = (problematic_tours['energy_consumption_kwh_cleaned'] - problematic_tours['energy_recharged_kwh']) - threshold
energy_public = required_energy.sum()
depot_charging = df_tours_energy_no_disp['energy_consumption_kwh_cleaned'] - df_tours_energy_no_disp['energy_recharged_kwh']
depot_charging = depot_charging.clip(upper=threshold)
energy_depot = depot_charging.sum()


total_energy = energy_industrial + energy_public + energy_depot
print(f"Total energy required from public charging: {energy_public:.2f} kWh")
print(f"Total energy required from industrial area charging: {energy_industrial:.2f} kWh")
print(f"Total energy required from depot charging: {energy_depot:.2f} kWh")

print('\n')
print(f"public charging percentage: {energy_public / total_energy * 100:.2f}%")
print(f"industrial area charging percentage: {energy_industrial / total_energy * 100:.2f}%")
print(f"depot charging percentage: {energy_depot / total_energy * 100:.2f}%")

### Daily Energy Demands per Home Base

In [ ]:
daily_demands = dp.daily_energy_demands(df_tours_energy_no_disp, threshold, charging_powers)

print(daily_demands.head())

# Most critical for ff 4. Where we have a representative sample of 10% of all of the ff's trucks --> daily load profiles would be 10x of these results
# 90/135 or 66 % of all recorded days could not be served with one traffo even if we assume an even load profile throughout the day 
# (max recharge in a day with 1 traffo = 630 kW * 24h = 15120 kWh)

### Charging Load Profiles per Home Base

In [ ]:
dv.plot_weekly_energy_demand_boxplot2(daily_demands)

## Create Load Profiles *

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import data.data_handling.load_profile as lp

print(charging_powers)
# Call the function with the load threshold parameter (default is 630 kW)
load_profiles, charging_stats = lp.calculate_charging_load_profiles(
    df_tours_energy_no_disp, charging_powers, threshold, load_threshold=630, save=True)

charging_durations = {cid: stats['avg_charging_duration_min'] for cid, stats in charging_stats.items()}

# If you want to display the statistics after calculation
for cid, stats in charging_stats.items():
    print(f"CID {cid}:")
    print(f"  Average charging duration: {stats['avg_charging_duration_min']:.2f} minutes")
    print(f"  Max load: {stats['max_load_kW']:.2f} kW")
    print(f"  Average time above threshold per day: {stats['avg_minutes_above_threshold']:.2f} minutes")
    print("\n")
# inputs: charging powers, df_tours_energy_no_disp

# intermediate dfs (one for each cid)
#   rows: minutes
#   columns: "charging package" (i.e energy demand / consumption of arriving track - charging power * time)
#            if no charging package then all values in row are None 
# -> load = number of charging packages each minute * charging power


# output one df per cid:
#   rows: minutes
#   columns: date, freight_forwarder, current load in kW

In [ ]:
# Call the function with all load profiles at once
stats = dv.plot_load_profiles_grid_paper(load_profiles, charging_powers, charging_durations) #TODO: change between 'abstract' and 'paper' version if required

# Print statistics for each CID
for cid, cid_stats in stats.items():
    print(f"\nCID {cid}:")
    print(f"  Number of active days: {cid_stats['active_days']}")
    print(f"  Average maximum load: {cid_stats['avg_max_load']:.2f} kW")
    if cid_stats['peak_time']:
        print(f"  Peak time: {cid_stats['peak_time']} (showing highest average load)")
    print(f"  Peak average load: {cid_stats['peak_avg_load']:.2f} kW")

In [ ]:
for idx, load_profile in load_profiles.items():
    print('\n')
    print(idx)
    stats = dv.plot_load_profile(load_profile, idx, charging_powers, charging_durations[idx])

    print(f"Number of active days: {stats['active_days']}")
    print(f"Average maximum load: {stats['avg_max_load']:.2f} kW")
    print(f"Peak time: {stats['peak_time']} (showing highest average load)")
    print(f"Peak average load: {stats['peak_avg_load']:.2f} kW")